# Bài 9: Nơ-ron nhân tạo
**Mở hộp đen AI · Tập 2 · Phòng thí nghiệm Colab**

Bốn chặng: **Chạy → Đọc → Sửa → Tự làm**.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 2. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Một nơ-ron: nhân, cộng, rồi quyết định

In [ ]:
import numpy as np

def noron(x, w, b):
    z = np.dot(w, x) + b          # tổng có trọng số
    return 1 if z > 0 else 0      # bậc thang: vượt ngưỡng thì "bật"

# Nơ-ron "đi chơi": x1 = trời đẹp, x2 = có bạn đi cùng
w = np.array([1, 1]); b = -1.5
for x in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    print(x, "->", noron(np.array(x), w, b))

### Ô 2. Dạy nơ-ron tự tìm trọng số (quy tắc học của perceptron)

In [ ]:
def hoc_perceptron(X, y, toc_do=1.0, so_vong=10, in_ra=False):
    w = np.zeros(X.shape[1]); b = 0.0
    for vong in range(so_vong):
        so_loi = 0
        for xi, yi in zip(X, y):
            du_doan = 1 if np.dot(w, xi) + b > 0 else 0
            sai = yi - du_doan                 # +1, 0 hoặc -1
            w = w + toc_do * sai * xi          # sai thì xoay đường thẳng
            b = b + toc_do * sai
            so_loi += abs(sai)
        if in_ra:
            print(f"vòng {vong + 1}: w = {w}, b = {b}, số lần sai = {so_loi}")
        if so_loi == 0:
            break
    return w, b

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_va = np.array([0, 0, 0, 1])                  # cổng VÀ (AND)
w, b = hoc_perceptron(X, y_va, in_ra=True)

### Ô 3. Bài toán không có đường thẳng nào chia được (XOR)

In [ ]:
y_xor = np.array([0, 1, 1, 0])                 # "khác nhau thì 1"
w, b = hoc_perceptron(X, y_xor, so_vong=100)
dung = sum(noron(xi, w, b) == yi for xi, yi in zip(X, y_xor))
print("Sau 100 vòng, đoán đúng", dung, "trên 4 trường hợp")

### Ô 4. Nhận chữ số viết tay: 0 hay 1?
Bộ dữ liệu `digits` có sẵn trong scikit-learn: mỗi ảnh 8 × 8 điểm, độ sáng từ 0 đến 16.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

so = load_digits()
chon = (so.target == 0) | (so.target == 1)
X01 = so.data[chon] / 16                       # đưa độ sáng về 0..1
y01 = so.target[chon]
X_hoc, X_thi, y_hoc, y_thi = train_test_split(X01, y01, test_size=0.3,
                                              random_state=0)
print("Số ảnh:", len(X01), "| mỗi ảnh có", X01.shape[1], "đặc trưng")

w, b = hoc_perceptron(X_hoc, y_hoc, toc_do=0.1, so_vong=20)
doan = np.array([noron(x, w, b) for x in X_thi])
print("Perceptron tự viết - đoán đúng trên đề thi:", round((doan == y_thi).mean(), 3))

### Ô 5. So với perceptron của thư viện

In [ ]:
from sklearn.linear_model import Perceptron

p = Perceptron(random_state=0).fit(X_hoc, y_hoc)
print("Perceptron của scikit-learn - đoán đúng trên đề thi:",
      round(p.score(X_thi, y_thi), 3))

## Chặng 2: Đọc
1. Trong ô 2, dòng nào là lúc nơ-ron **học**? Khi đoán đúng thì trọng số có đổi không?
2. Vì sao ô 3 không bao giờ đúng cả 4 trường hợp?
3. Ở ô 4, nơ-ron có 64 trọng số. Mỗi trọng số ứng với cái gì?

## Chặng 3: Sửa
### Ô 6. Xem nơ-ron đã học được gì: vẽ 64 trọng số thành ảnh 8 × 8

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(w.reshape(8, 8), cmap="RdBu")
plt.colorbar(); plt.title("Đỏ: nghiêng về số 0 | Xanh: nghiêng về số 1"); plt.show()

### Ô 7. Thử với cặp khó nhất: 8 hay 9?

In [ ]:
chon = (so.target == 8) | (so.target == 9)
X38 = so.data[chon] / 16
y38 = (so.target[chon] == 9).astype(int)       # 1 là số 9, 0 là số 8
Xh, Xt, yh, yt = train_test_split(X38, y38, test_size=0.3, random_state=0)
w38, b38 = hoc_perceptron(Xh, yh, toc_do=0.1, so_vong=20)
doan = np.array([noron(x, w38, b38) for x in Xt])
print("8 hay 9 - đoán đúng trên đề thi:", round((doan == yt).mean(), 3))

## Chặng 4: Tự làm
1. Sửa ô 2 để dạy cổng HOẶC (OR): đầu ra là 1 nếu ít nhất một đầu vào bằng 1. Cần mấy vòng?
2. Ở ô 4, đổi `toc_do` thành 1.0 và 0.01. Kết quả có đổi không? Vì sao?
3. Chọn một cặp chữ số mà em nghĩ là khó nhất, thử ở ô 7 và giải thích.

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*